In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "ai":
    PROJECT_ROOT = PROJECT_ROOT.parent

route = PROJECT_ROOT / "etl" / "embedding" / "후처리_데이터"

AXES = ["EI", "NS", "FT", "JP"]
POSITIVE_LABELS = {"EI": "I", "NS": "N", "FT": "F", "JP": "J"}
RANDOM_STATE = 42

route

In [ ]:
X_train = np.load(route / "train_embeddings.npy")
X_test = np.load(route / "test_embeddings.npy")
X_val = np.load(route / "validation_embeddings.npy")

y_train = pd.read_csv(route / "train_labels.csv")
y_test = pd.read_csv(route / "test_labels.csv")
y_val = pd.read_csv(route / "validation_labels.csv")

assert len(X_train) == len(y_train), "train embeddings/labels row mismatch"
assert len(X_val) == len(y_val), "validation embeddings/labels row mismatch"
assert len(X_test) == len(y_test), "test embeddings/labels row mismatch"

print("train:", X_train.shape, y_train.shape)
print("validation:", X_val.shape, y_val.shape)
print("test:", X_test.shape, y_test.shape)

In [ ]:
y_train.head()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from xgboost import XGBClassifier


def make_model(scale_pos_weight=1.0):
    return XGBClassifier(
        n_estimators=120,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.5,
        min_child_weight=5,
        reg_alpha=1.0,
        reg_lambda=5.0,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def encode_axis(y, axis):
    positive_label = POSITIVE_LABELS[axis]
    negative_label = [label for label in sorted(y[axis].unique()) if label != positive_label][0]
    classes = np.array([negative_label, positive_label])
    encoded = (y[axis] == positive_label).astype(int)
    return encoded, classes


def get_scale_pos_weight(y_encoded):
    negative_count = (y_encoded == 0).sum()
    positive_count = (y_encoded == 1).sum()
    return negative_count / positive_count


def decode_binary_preds(preds, classes):
    return pd.Series(classes[np.asarray(preds).astype(int)])


def predict_with_threshold(proba, classes, threshold):
    return pd.Series(np.where(proba >= threshold, classes[1], classes[0]))


def get_positive_proba(model, X):
    return model.predict_proba(X)[:, 1]


def get_binary_true(y, axis):
    return (y[axis] == POSITIVE_LABELS[axis]).astype(int)


def find_best_threshold(y_true, proba, classes):
    best_threshold = 0.5
    best_f1 = -1
    for threshold in np.arange(0.2, 0.801, 0.01):
        preds = predict_with_threshold(proba, classes, threshold)
        score = f1_score(y_true, preds, average="macro")
        if score > best_f1:
            best_f1 = score
            best_threshold = threshold
    return best_threshold, best_f1


def fit_axis_model(axis):
    y_train_encoded, classes = encode_axis(y_train, axis)
    scale_pos_weight = get_scale_pos_weight(y_train_encoded)
    model = make_model(scale_pos_weight=scale_pos_weight)
    model.fit(
        X_train,
        y_train_encoded,
        eval_set=[(X_val, get_binary_true(y_val, axis))],
        verbose=False,
    )
    proba = get_positive_proba(model, X_val)
    threshold, threshold_f1 = find_best_threshold(y_val[axis], proba, classes)
    preds = predict_with_threshold(proba, classes, threshold)
    print(axis, classes, "scale_pos_weight=", round(scale_pos_weight, 3), "threshold=", round(threshold, 2), "macro_f1=", round(threshold_f1, 4))
    print(proba[:5])
    return model, classes, preds, proba, threshold, scale_pos_weight


model_EI, classes_EI, preds_EI, proba_EI, threshold_EI, scale_pos_weight_EI = fit_axis_model("EI")

In [ ]:
model_NS, classes_NS, preds_NS, proba_NS, threshold_NS, scale_pos_weight_NS = fit_axis_model("NS")

In [ ]:
model_FT, classes_FT, preds_FT, proba_FT, threshold_FT, scale_pos_weight_FT = fit_axis_model("FT")

In [ ]:
model_JP, classes_JP, preds_JP, proba_JP, threshold_JP, scale_pos_weight_JP = fit_axis_model("JP")

In [ ]:
# EI	NS	FT	JP


thresholds = {
    "EI": threshold_EI,
    "NS": threshold_NS,
    "FT": threshold_FT,
    "JP": threshold_JP,
}

scale_pos_weights = {
    "EI": scale_pos_weight_EI,
    "NS": scale_pos_weight_NS,
    "FT": scale_pos_weight_FT,
    "JP": scale_pos_weight_JP,
}

validation_results = []
for axis, preds, proba in [
    ("EI", preds_EI, proba_EI),
    ("NS", preds_NS, proba_NS),
    ("FT", preds_FT, proba_FT),
    ("JP", preds_JP, proba_JP),
]:
    y_true_binary = get_binary_true(y_val, axis)
    validation_results.append(
        {
            "axis": axis,
            "positive_label": POSITIVE_LABELS[axis],
            "scale_pos_weight": scale_pos_weights[axis],
            "threshold": thresholds[axis],
            "accuracy": accuracy_score(y_val[axis], preds),
            "f1_macro": f1_score(y_val[axis], preds, average="macro"),
            "roc_auc": roc_auc_score(y_true_binary, proba),
        }
    )

validation_results = pd.DataFrame(validation_results)
validation_results

In [ ]:
majority_baseline = []
for axis in AXES:
    majority_accuracy = y_val[axis].value_counts(normalize=True).max()
    model_accuracy = validation_results.loc[validation_results["axis"] == axis, "accuracy"].iloc[0]
    majority_baseline.append(
        {
            "axis": axis,
            "majority_baseline_accuracy": majority_accuracy,
            "xgboost_accuracy": model_accuracy,
            "accuracy_lift": model_accuracy - majority_accuracy,
        }
    )

majority_baseline = pd.DataFrame(majority_baseline)
majority_baseline

In [ ]:
prediction_distribution = []
for axis, preds in [
    ("EI", preds_EI),
    ("NS", preds_NS),
    ("FT", preds_FT),
    ("JP", preds_JP),
]:
    true_ratio = y_val[axis].value_counts(normalize=True).sort_index()
    pred_ratio = pd.Series(preds).value_counts(normalize=True).sort_index()
    for label in sorted(y_val[axis].unique()):
        prediction_distribution.append(
            {
                "axis": axis,
                "label": label,
                "true_ratio": true_ratio.get(label, 0),
                "pred_ratio": pred_ratio.get(label, 0),
                "pred_minus_true": pred_ratio.get(label, 0) - true_ratio.get(label, 0),
            }
        )

prediction_distribution = pd.DataFrame(prediction_distribution)
prediction_distribution

In [ ]:
model_map = {"EI": model_EI, "NS": model_NS, "FT": model_FT, "JP": model_JP}
classes_map = {"EI": classes_EI, "NS": classes_NS, "FT": classes_FT, "JP": classes_JP}
threshold_map = {"EI": threshold_EI, "NS": threshold_NS, "FT": threshold_FT, "JP": threshold_JP}

train_validation_results = []
for split_name, X, y in [("train", X_train, y_train), ("validation", X_val, y_val)]:
    for axis in AXES:
        model = model_map[axis]
        classes = classes_map[axis]
        proba = get_positive_proba(model, X)
        preds = predict_with_threshold(proba, classes, threshold_map[axis])
        train_validation_results.append(
            {
                "split": split_name,
                "axis": axis,
                "accuracy": accuracy_score(y[axis], preds),
                "f1_macro": f1_score(y[axis], preds, average="macro"),
                "roc_auc": roc_auc_score(get_binary_true(y, axis), proba),
            }
        )

train_validation_results = pd.DataFrame(train_validation_results)
train_validation_results

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(get_binary_true(y_val, "EI"), proba_EI, name="EI", ax=ax)
RocCurveDisplay.from_predictions(get_binary_true(y_val, "NS"), proba_NS, name="NS", ax=ax)
RocCurveDisplay.from_predictions(get_binary_true(y_val, "FT"), proba_FT, name="FT", ax=ax)
RocCurveDisplay.from_predictions(get_binary_true(y_val, "JP"), proba_JP, name="JP", ax=ax)
ax.set_title("Validation ROC Curve")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
for axis, preds in [
    ("EI", preds_EI),
    ("NS", preds_NS),
    ("FT", preds_FT),
    ("JP", preds_JP),
]:
    print(f"[{axis}] classification report")
    print(classification_report(y_val[axis], preds))

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrices = {}
for axis, preds in [
    ("EI", preds_EI),
    ("NS", preds_NS),
    ("FT", preds_FT),
    ("JP", preds_JP),
]:
    labels = sorted(y_val[axis].unique())
    confusion_matrices[axis] = pd.DataFrame(
        confusion_matrix(y_val[axis], preds, labels=labels, normalize="true"),
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for ax, axis in zip(axes.ravel(), AXES):
    sns.heatmap(confusion_matrices[axis], annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(axis)
plt.tight_layout()
plt.show()

confusion_matrices["EI"]

In [ ]:
confusion_matrices["NS"]

In [ ]:
confusion_matrices["FT"]

In [ ]:
confusion_matrices["JP"]

In [ ]:
import joblib

models = {
    "EI": model_EI,
    "NS": model_NS,
    "FT": model_FT,
    "JP": model_JP,
}

model_classes = {
    "EI": classes_EI,
    "NS": classes_NS,
    "FT": classes_FT,
    "JP": classes_JP,
}

# 필요할 때만 주석을 해제해서 모델을 저장하세요.
# joblib.dump(
#     {
#         "models": models,
#         "model_classes": model_classes,
#         "positive_labels": POSITIVE_LABELS,
#         "thresholds": thresholds,
#         "scale_pos_weights": scale_pos_weights,
#     },
#     PROJECT_ROOT / "ai" / "mbti_axis_xgboost_models.joblib",
# )